### 二、以下是多推理谓词的结构优先的整体流程示例

#### 1. 将 CSV 数据转换为 GraphLib 格式，这个格式中边是带标签的:

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import re
import sys
import math
import time
import tempfile
import traceback
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np
import json
from pythonProject.src.Structure_first.fastest_pipeline import FastestGraphConverter, FastestEstimateMerger
from pythonProject.src.Structure_first.graph_sample import FastestRunner
from pythonProject.src.Structure_first.precision_submatching import ExactSubgraphMatcher
from pythonProject.src.Structure_first.proxy_sample import ProxyStratifiedSampler, compute_T_true

# 一级测试数据集
datasets_name = "parler_data"
# 一级数据集下根据查询和图结构的差异划分的子测试数据集
# dataset_name = "dataset_test"
dataset_name = "dataset_test"
# 原始CSV数据路径
CSV_BASE_DIR = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/csv_data"
# 转换后GraphLib数据存放路径
Graph_Lib_Dir = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/data_graph"
# converter = FastestGraphConverter(CSV_BASE_DIR,Graph_Lib_Dir)
# converter.run_without_author_user_post()
# converter.remove_edge_labels()

#### 2.1同时调用准确的子图匹配算法得到真实结果的基数，这些准确子图匹配算法只能处理边不带标签的图

##### 2.1.1 首先基于原来边带标签的图，精简成边不带标签的图

In [ ]:
# converter = FastestGraphConverter(CSV_BASE_DIR,Graph_Lib_Dir)
# converter.simplify_graph_merge_edges_update_degree(input_path=Graph_Lib_Dir+'/parler.graph',
#                                                    output_path='/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/data_graph/parler.graph')

##### 2.1.2 然后对精简后的数据图调用准确子图匹配算法进行匹配，这里得到的GT对应的查询图，应该与Fastest对应的查询图等价（就算边没有标签也没有影响，且两点不存在多条边）

                                        六千万基数： 准确匹配需要2min左右
                                        1.2亿基数： 准确匹配需要4min左右
                                        2.4亿基数： 准确匹配需要8min左右
                                        10亿基数：  准确匹配需要40min左右

In [ ]:
matcher = ExactSubgraphMatcher(
    exe_path="/home/wangshuo/projects/SubgraphMatching/build/matching/SubgraphMatching.out",
    default_args=["-filter", "GQL", "-order", "GQL", "-engine", "LFTJ", "-num", "MAX"],
    timeout=3000,
)
matcher.run_batch(
    data_graph=f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph/parler.graph",
    query_graph_dir=f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/query_graph",
    output_dir=f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/ground_truth",
)

#### 2.2 基于GraphLib数据，调用Fastest进行树采样或图采样，得到特定标签下所有或部分点的估计值:
（1) 对于随机生成的查询可以先调用Fastest估计基数，选择基数合适的查询作为实验对象 --- 对于parler_ans.txt中的数据可以先随机生成
（2）正常流畅下，需要先调用精确的子图匹配算法，得到真实基数结果存放到parler_ans.txt，然后再调用Fastest进行估计

In [ ]:
# dataset_name = "dataset_one"
runner = FastestRunner(build_dir="/home/wangshuo/projects/FaSTest-main/build")
print(dataset_name)
sample_budget = 10000
# 默认执行 ./Fastest -d parler --ROOT_LABEL 1 (表示推理谓词所在节点的标签)
code, output = runner.run(dataset=dataset_name, root_label=-1,sample_budget=sample_budget)

##### 2.2.1 拆分多谓词多查询的结果文件 ins_estimateW_result.csv 为多谓词单查询的结果文件，命名格式为{query_name}_estimateW_result.csv

In [ ]:
import pandas as pd
import os

def split_results_by_query(input_file_path, output_dir):
    """
    拆分多查询的结果文件为单查询的结果文件。

    Args:
        input_file_path (str): 输入的 ins_estimateW_result.csv 文件路径。
        output_dir (str): 拆分后的小文件要保存的目录。
    """
    print(f"--- 开始处理文件: {input_file_path} ---")

    # 检查输入文件是否存在
    if not os.path.exists(input_file_path):
        print(f"[错误] 输入文件不存在: {input_file_path}")
        return

    # 检查输出目录是否存在，如果不存在则创建
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"创建输出目录: {output_dir}")

    try:
        # 使用 Pandas 读取整个 CSV 文件
        df = pd.read_csv(input_file_path)
        print(f"成功读取 {len(df)} 行数据。")

        # 检查 'query_name' 列是否存在
        if 'query_name' not in df.columns:
            print("[错误] CSV文件中缺少 'query_name' 列。请检查文件格式。")
            return

        # 按照 'query_name' 列的值进行分组
        grouped = df.groupby('query_name')
        
        num_files_created = 0
        for query_name, group_df in grouped:
            # 1. 构造输出文件名
            base_name = os.path.splitext(query_name)[0]
            
            # +++ 修改下面这行，将后缀名从 .txt 改为 .csv +++
            output_filename = f"{base_name}.csv"
            
            output_filepath = os.path.join(output_dir, output_filename)
            
            print(f"  正在处理查询 '{query_name}' -> 输出到 '{output_filepath}'...")

            # 2. 准备要写入的数据
            output_df = group_df.drop('query_name', axis=1)

            # 3. 将处理后的数据写入新的 CSV 文件
            output_df.to_csv(output_filepath, index=False)
            
            num_files_created += 1

        print(f"\n--- 处理完成 ---")
        print(f"总共拆分出 {num_files_created} 个查询文件。")

    except Exception as e:
        print(f"[严重错误] 处理过程中发生异常: {e}")

# ===================================================================
# --- Jupyter Notebook 配置区 ---
# 请在这里修改您的输入文件路径和输出目录路径
# ===================================================================

# 假设您的数据集名称是 'dataset_test'
dataset_name = 'dataset_test'

# 构造输入文件和输出目录的路径
# 请确保这里的路径与您的实际文件结构一致
base_path = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/results/"
input_csv_path = os.path.join(base_path, "ins_estimateW_result.csv")
# 将拆分后的文件保存到 'structure_estimate' 子目录中
output_directory = os.path.join(base_path, "structure_estimate/")

# 调用主函数
split_results_by_query(input_csv_path, output_directory)

# ===================================================================

/home/wangshuo/resource/datasets/parler_data/dataset_test/results/structure_estimate 
1.读取上面文件夹下的所有文件，统计instance_id 不同的个数是多少，也就是instance_id max值，global_estimateW，按照global_estimateW 降序排列。
2.输出global_estimateW < 100000000 and instance_id 不同的个数 count > 1000的文件列表

In [ ]:
import os
import pandas as pd
from typing import List, Dict # 兼容 Python 3.8

def analyze_estimate_files(directory_path: str):
    """
    统计指定目录下所有CSV文件，收集实例数(count)和global_estimateW，
    按 global_estimateW 降序排列，并筛选出满足特定条件的文件列表及其相关值。

    参数:
        directory_path (str): 包含查询结果CSV文件的目录路径。
    """
    print(f"====== 开始分析目录: {directory_path} ======")

    # 1. 检查目录是否存在
    if not os.path.isdir(directory_path):
        print(f"[错误] 目录不存在: {directory_path}")
        return

    # 2. 查找所有相关的 .csv 文件
    try:
        csv_files = sorted([f for f in os.listdir(directory_path) if f.endswith('.csv')])
    except FileNotFoundError:
        print(f"[错误] 无法访问目录: {directory_path}")
        return

    if not csv_files:
        print(f"在该目录中没有找到任何 .csv 文件。")
        return
        
    print(f"找到 {len(csv_files)} 个 CSV 文件进行分析...\n")

    # 3. 遍历每个文件进行统计
    results = []
    
    for filename in csv_files:
        filepath = os.path.join(directory_path, filename)
        
        try:
            df = pd.read_csv(filepath)
            
            required_cols = ['instance_id', 'global_estimateW']
            if not all(col in df.columns for col in required_cols):
                print(f"  - 文件 '{filename}': [警告] 缺少 'instance_id' 或 'global_estimateW' 列，已跳过。")
                continue
                
            if df.empty:
                unique_count = 0
                global_estimate_w = 0.0
            else:
                unique_count = df['instance_id'].nunique()
                global_estimate_w = df['global_estimateW'].iloc[0]

            results.append({
                '文件名': filename,
                '实例数(count)': unique_count,
                'global_estimateW': global_estimate_w,
            })
            
        except Exception as e:
            print(f"  - 文件 '{filename}': [错误] 处理时发生异常: {e}")
    
    # 4. 生成、排序并打印总结报告
    if not results:
        print("\n未能从任何文件中成功提取统计信息。")
        return
        
    report_df = pd.DataFrame(results)
    
    # 按 global_estimateW 降序排列
    report_df = report_df.sort_values(by='global_estimateW', ascending=False).reset_index(drop=True)
    
    print("=" * 80)
    print("====== 查询统计总结 (按 global_estimateW 降序排列) ======")
    with pd.option_context('display.max_rows', None, 'display.width', 120):
        print(report_df.to_string(index=False))
    print("=" * 80)

    # 5. 筛选并打印满足条件的文件列表及其相关值
    print("\n" + "="*80)
    print("====== 满足条件的文件列表 (global_estimateW < 100,000,000 and 实例数 > 1,000) ======")
    
    filtered_df = report_df[
        (report_df['global_estimateW'] < 100000000) & 
        (report_df['实例数(count)'] > 1000)
    ].copy()
    print(len(filtered_df))
    if filtered_df.empty:
        print("没有找到满足条件的文件。")
    else:
        print("以下文件满足筛选条件:")
        # 只打印筛选后的三列，并使用 to_string 格式化输出
        print(filtered_df[['文件名', 'global_estimateW', '实例数(count)']].to_string(index=False))
            
    print("=" * 80)
    print(f'[info]: {len(filtered_df)}')
    print("\n[完成] 分析结束。")


if __name__ == '__main__':
    # --- 配置区 ---
    DATASET = 'dataset_test'
    
    # 自动构建目标目录路径
    TARGET_DIRECTORY = f"/home/wangshuo/resource/datasets/parler_data/{DATASET}/results/structure_estimate"
    
    # --- 执行 ---
    analyze_estimate_files(TARGET_DIRECTORY)

3.将不满足这个条件global_estimateW < 100000000 and instance_id 不同的个数 count > 1000的文件列表的所有查询，保存到同级文件夹difficult_query中去

In [ ]:
import os
import pandas as pd
import shutil
from typing import List, Dict, Optional # 兼容 Python 3.8

def analyze_files(directory_path: str) -> Optional[pd.DataFrame]:
    """
    分析目录下的所有CSV文件，并返回一个包含统计信息的DataFrame。
    """
    print(f"====== 开始分析目录: {directory_path} ======")

    if not os.path.isdir(directory_path):
        print(f"[错误] 目录不存在: {directory_path}")
        return None

    try:
        csv_files = sorted([f for f in os.listdir(directory_path) if f.endswith('.csv')])
    except FileNotFoundError:
        print(f"[错误] 无法访问目录: {directory_path}")
        return None

    if not csv_files:
        print("在该目录中没有找到任何 .csv 文件。")
        return None
        
    print(f"找到 {len(csv_files)} 个 CSV 文件进行分析...\n")

    results = []
    for filename in csv_files:
        filepath = os.path.join(directory_path, filename)
        try:
            df = pd.read_csv(filepath)
            
            required_cols = ['instance_id', 'global_estimateW']
            if not all(col in df.columns for col in required_cols):
                print(f"  - 文件 '{filename}': [警告] 缺少 'instance_id' 或 'global_estimateW' 列，已跳过。")
                continue
            
            unique_count = df['instance_id'].nunique() if not df.empty else 0
            global_estimate_w = df['global_estimateW'].iloc[0] if not df.empty else 0.0

            results.append({
                '文件名': filename,
                '实例数(count)': unique_count,
                'global_estimateW': global_estimate_w,
            })
        except Exception as e:
            print(f"  - 文件 '{filename}': [错误] 处理时发生异常: {e}")
    
    if not results:
        return None
        
    report_df = pd.DataFrame(results)
    report_df = report_df.sort_values(by='global_estimateW', ascending=False).reset_index(drop=True)
    return report_df


def move_queries(
    files_to_move_df: pd.DataFrame, 
    source_query_dir: str, 
    destination_dir: str
):
    """
    将指定的查询文件从源目录【移动】到目标目录 (修复版)。
    """
    if files_to_move_df.empty:
        print("\n没有需要移动的 '困难' 查询。")
        return

    print(f"\n====== 开始移动 '困难' 查询到: {destination_dir} ======")
    os.makedirs(destination_dir, exist_ok=True)

    moved_count = 0
    # files_to_move_df['文件名'] 中包含的是类似 'query_cycle_8_45.csv' 的字符串
    for csv_filename in files_to_move_df['文件名']:
        
        # --- 【核心修复】在这里 ---
        # 确保输入的文件名确实是以 .csv 结尾
        if not csv_filename.endswith('.csv'):
            print(f"  - [警告] 文件名 '{csv_filename}' 不以 .csv 结尾，无法推断 .graph 文件名，已跳过。")
            continue
            
        # 正确的转换逻辑：直接将 .csv 后缀替换为 .graph
        graph_filename = csv_filename.replace('.csv', '.graph')
        
        source_path = os.path.join(source_query_dir, graph_filename)
        destination_path = os.path.join(destination_dir, graph_filename)
        
        if os.path.exists(source_path):
            try:
                shutil.move(source_path, destination_path)
                moved_count += 1
            except Exception as e:
                print(f"  - [错误] 移动文件 {graph_filename} 时失败: {e}")
        else:
            print(f"  - [警告] 未找到源文件，无法移动: {source_path}")
            
    print(f"✅ 移动完成，总共移动了 {moved_count} 个查询文件。")


def main():
    """主执行函数"""
    # --- 配置区 ---
    DATASET = 'dataset_test'
    
    # --- 路径配置 ---
    base_data_path = f"/home/wangshuo/resource/datasets/parler_data/{DATASET}"
    
    # 输入目录 (包含要分析的CSV)
    analysis_dir = os.path.join(base_data_path, "results", "structure_estimate")
    
    # 原始查询图所在的目录
    source_query_dir = os.path.join(base_data_path, "query_graph")
    
    # “困难”查询的目标目录 (与 analysis_dir 同级)
    difficult_query_dir = os.path.join(base_data_path, "results", "difficult_query")

    # --- 1. 执行分析并获取报告 ---
    report_df = analyze_files(analysis_dir)

    if report_df is None:
        print("\n分析未能生成报告，程序终止。")
        return

    print("\n" + "=" * 80)
    print("====== 查询统计总结 (按 global_estimateW 降序排列) ======")
    with pd.option_context('display.max_rows', None, 'display.width', 120):
        print(report_df.to_string(index=False))
    print("=" * 80)
    
    # --- 2. 筛选出“简单”和“困难”的查询 ---
    
    # 条件1: global_estimateW < 100,000,000
    cond1 = report_df['global_estimateW'] < 100000000
    # 条件2: 实例数(count) > 1,000
    cond2 = report_df['实例数(count)'] > 1000
    
    easy_queries_df = report_df[cond1 & cond2].copy()
    # “困难”查询是“简单”查询的补集
    difficult_queries_df = report_df[~(cond1 & cond2)].copy()

    # --- 3. 打印筛选结果 ---
    print("\n" + "="*80)
    print("====== 满足条件的 '简单' 查询列表 ======")
    if easy_queries_df.empty:
        print("没有找到满足条件的 '简单' 查询。")
    else:
        print(easy_queries_df[['文件名', 'global_estimateW', '实例数(count)']].to_string(index=False))
    print("=" * 80)

    print("\n" + "="*80)
    print("====== 不满足条件的 '困难' 查询列表 ======")
    if difficult_queries_df.empty:
        print("所有查询都满足 '简单' 条件，没有 '困难' 查询。")
    else:
        print(difficult_queries_df[['文件名', 'global_estimateW', '实例数(count)']].to_string(index=False))
    print("=" * 80)

    # --- 4. 移动“困难”查询的文件 ---
    move_queries(
        files_to_move_df=difficult_queries_df,
        source_query_dir=source_query_dir,
        destination_dir=difficult_query_dir
    )
    
    print("\n[全部完成]")


if __name__ == '__main__':
    main()

### 3 - 4:
"""
处理 multi-query 的 Fastest 输出(in_estimateW_result.txt)：
  - 解析多个 Query 的每节点估计值（支持多个 Query 块）
  - 根据 INFER_NODE_FILE 中每行指定的 gt_match_col（例如 u1,u2）按顺序对应每个 Query
  - 将每个 Query 的 estimate 列并入 post_with_estimate.csv（列名 estimate__<query_basename>）
  - 针对每个 Query 用 ProxyStratifiedSampler 做评估（使用 compute_T_true 得到 T_true）
  - 输出 summary CSV / TXT

"""

#### 3-1 多推理谓词核心集的实例对估计值与csvjoin得到推理谓词概率，用于第二步采样
instance_id,estimateW,global_estimateW,ML1_oracle1_probability,ML1_proxy4b1_probability,ML1_proxy2b1_probability,ML2_oracle2_probability,ML2_proxy2d2_probability,ML2_proxy4d2_probability
1,12.9165,129165.0,['0.013562889'],['0.0002797'],['0.000718'],['0.992257595062256'],['0.74853515625'],['0.400146484375']
2,12.9165,129165.0,['0.013562889'],['0.0002797'],['0.000718'],['0.9998629093170166'],['0.794921875'],['0.80419921875']
3,6.4582,129165.0,['0.013562889'],['0.0002797'],['0.000718'],['0.9998335838317872'],['0.7841796875'],['0.8037109375']

In [ ]:
import pandas as pd
import os
from typing import Dict

# ... (load_and_prepare_mappings 和 load_source_csvs 函数保持不变) ...
def load_and_prepare_mappings(id_mapping_path: str) -> pd.DataFrame:
    """读取id_mapping.csv并准备好用于连接的DataFrame。"""
    if not os.path.exists(id_mapping_path):
        raise FileNotFoundError(f"ID映射文件不存在: {id_mapping_path}")
        
    id_map_df = pd.read_csv(id_mapping_path, dtype={'internal_id': str, 'orig_id': str, 'type': str})
    id_map_df.rename(columns={'internal_id': 'node_id'}, inplace=True)
    
    print(f"加载了 {len(id_map_df)} 条ID映射记录。")
    return id_map_df

def load_source_csvs(post_csv_path: str, comment_csv_path: str) -> Dict[str, pd.DataFrame]:
    """读取原始的 post.csv 和 comment.csv 文件。"""
    if not os.path.exists(post_csv_path):
        raise FileNotFoundError(f"post.csv 文件不存在: {post_csv_path}")
    if not os.path.exists(comment_csv_path):
        raise FileNotFoundError(f"comment.csv 文件不存在: {comment_csv_path}")
        
    post_df = pd.read_csv(post_csv_path, dtype=str)
    comment_df = pd.read_csv(comment_csv_path, dtype=str)
    
    if 'id:ID' in post_df.columns:
        post_df.rename(columns={'id:ID': 'orig_id'}, inplace=True)
    if 'id:ID' in comment_df.columns:
        comment_df.rename(columns={'id:ID': 'orig_id'}, inplace=True)
        
    print(f"加载了 {len(post_df)} 行 post 数据和 {len(comment_df)} 行 comment 数据。")
    return {"Post": post_df, "Comment": comment_df}


def process_single_query_file_correctly(query_file_path: str, id_map_df: pd.DataFrame, sources: Dict[str, pd.DataFrame], output_dir: str):
    """
    处理单个长格式查询文件，聚合 ML 值和【节点ID列表】。
    """
    query_basename = os.path.basename(query_file_path).replace("_estimateW_result.csv", "")
    print(f"\n--- 正在处理查询: {query_basename} ---")
    
    instance_df = pd.read_csv(query_file_path)
    if instance_df.empty:
        print("文件为空，跳过。")
        return
        
    instance_df['node_id'] = instance_df['node_id'].astype(str)
    
    # 1. 连接ID映射
    merged_with_map = pd.merge(instance_df, id_map_df, on='node_id', how='left')

    # 2. 定义期望的 ML 列
    expected_ml1_cols = ['ML1_oracle1_probability', 'ML1_proxy4b1_probability', 'ML1_proxy2b1_probability']
    expected_ml2_cols = ['ML2_oracle2_probability', 'ML2_proxy2d2_probability', 'ML2_proxy4d2_probability', 'ML2_proxy1_probability']
    
    # 3. 分别处理 Post 和 Comment 数据流
    
    # --- Post 数据流 ---
    posts_data = merged_with_map[merged_with_map['type'] == 'Post'].copy()
    posts_joined = pd.merge(posts_data, sources['Post'], on='orig_id', how='left')
    
    # --- Comment 数据流 ---
    comments_data = merged_with_map[merged_with_map['type'] == 'Comment'].copy()
    comments_joined = pd.merge(comments_data, sources['Comment'], on='orig_id', how='left')
    
    # 4. 最终组装与聚合
    
    # --- 分别对 Post 和 Comment 进行聚合 ---
    
    # === 【修改点 1】：聚合 Post 数据（增加 orig_id） ===
    actual_ml1_cols = [col for col in expected_ml1_cols if col in posts_joined.columns]
    if not posts_joined.empty:
        # 定义聚合字典：ML列聚合为列表，orig_id 也聚合为列表
        agg_dict = {col: list for col in actual_ml1_cols}
        agg_dict['orig_id'] = list  # +++ 新增：收集节点ID +++
        
        agg_posts = posts_joined.groupby('instance_id').agg(agg_dict).reset_index()
        # 重命名 orig_id 为 post_id_list
        agg_posts.rename(columns={'orig_id': 'post_id_list'}, inplace=True) # +++ 重命名 +++
    else:
        # 如果为空，创建带有所需列的空 DataFrame
        agg_posts = pd.DataFrame(columns=['instance_id', 'post_id_list'] + actual_ml1_cols)

    # === 【修改点 2】：聚合 Comment 数据（增加 orig_id） ===
    actual_ml2_cols = [col for col in expected_ml2_cols if col in comments_joined.columns]
    if not comments_joined.empty:
        agg_dict = {col: list for col in actual_ml2_cols}
        agg_dict['orig_id'] = list  # +++ 新增：收集节点ID +++
        
        agg_comments = comments_joined.groupby('instance_id').agg(agg_dict).reset_index()
        # 重命名 orig_id 为 comment_id_list
        agg_comments.rename(columns={'orig_id': 'comment_id_list'}, inplace=True) # +++ 重命名 +++
    else:
        agg_comments = pd.DataFrame(columns=['instance_id', 'comment_id_list'] + actual_ml2_cols)
        
    # 创建基础表
    base_agg_df = instance_df[['instance_id', 'estimateW', 'global_estimateW']].groupby('instance_id').first().reset_index()
    
    # --- 合并聚合后的结果 ---
    final_df = pd.merge(base_agg_df, agg_posts, on='instance_id', how='left')
    final_df = pd.merge(final_df, agg_comments, on='instance_id', how='left')

    print(f"聚合完成，生成 {len(final_df)} 条实例记录。")

    # 5. 保存结果
    output_filename = f"aggregated_list_{query_basename}.csv"
    output_filepath = os.path.join(output_dir, output_filename)
    
    # === 【修改点 3】：更新最终列顺序，加入 ID 列表列 ===
    final_columns_order = [
        'instance_id', 'estimateW', 'global_estimateW', 
        'post_id_list', 'comment_id_list' # +++ 确保这两列被包含 +++
    ] + expected_ml1_cols + expected_ml2_cols
    
    final_df = final_df.reindex(columns=final_columns_order)
    
    final_df.to_csv(output_filepath, index=False)
    print(f"[完成] 结果已保存到: {output_filepath}")
def main(dataset_name):
    """主执行函数"""
    print(f"====== 开始处理数据集: {dataset_name} ======")
    
    # --- 路径配置 ---
    base_path = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}"
    estimate_dir = os.path.join(base_path, "results", "structure_estimate")
    id_mapping_path = os.path.join(base_path, "data_graph", "id_mapping.csv")
    post_csv_path = os.path.join(base_path, "csv_data", "post.csv")
    comment_csv_path = os.path.join(base_path, "csv_data", "comment.csv")
    output_dir = os.path.join(base_path, "results", "aggregated_results")
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"创建输出目录: {output_dir}")

    # --- 执行流程 ---
    try:
        id_map_df = load_and_prepare_mappings(id_mapping_path)
        sources = load_source_csvs(post_csv_path, comment_csv_path)

        query_files = [f for f in os.listdir(estimate_dir) if f.endswith('.csv')]
        if not query_files:
            print(f"[警告] 在目录 {estimate_dir} 中没有找到任何 .csv 结果文件。")
            return
            
        for query_file in sorted(query_files):
            query_file_path = os.path.join(estimate_dir, query_file)
            # 调用新的、正确的处理函数
            process_single_query_file_correctly(query_file_path, id_map_df, sources, output_dir)
            
        print(f"\n====== 数据集 {dataset_name} 处理完毕 ======")

    except FileNotFoundError as e:
        print(f"[严重错误] 依赖文件未找到: {e}")
    except Exception as e:
        print(f"[严重错误] 处理过程中发生未知异常: {e}")

if __name__ == '__main__':
    # --- Jupyter Notebook 或脚本配置区 ---
    dataset_name_to_process = 'dataset_test'
    main(dataset_name_to_process)

##### 3.1.1 验证程序正确性

In [ ]:
import pandas as pd
import os
import numpy as np

def verify_estimates_consistency(dataset_name: str):
    """
    验证 aggregated_results 目录下的每个CSV文件，
    检查其中 estimateW 列的总和是否近似等于 global_estimateW。
    """
    print(f"====== 开始验证数据集: {dataset_name} ======")

    # --- 路径配置 ---
    base_path = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}"
    aggregated_dir = os.path.join(base_path, "results", "aggregated_results")
    
    # 输出验证报告的路径
    report_path = os.path.join(base_path, "results", "consistency_report.txt")

    # --- 1. 检查输入目录是否存在 ---
    if not os.path.exists(aggregated_dir):
        print(f"[错误] 聚合结果目录不存在: {aggregated_dir}")
        return

    # --- 2. 遍历并处理每个聚合后的文件 ---
    agg_files = [f for f in os.listdir(aggregated_dir) if f.startswith('aggregated_list_') and f.endswith('.csv')]
    
    if not agg_files:
        print(f"[警告] 在目录 {aggregated_dir} 中没有找到任何 'aggregated_list_*.csv' 文件。")
        return
        
    print(f"找到 {len(agg_files)} 个聚合文件进行验证...\n")
    
    # 用于存储所有结果的列表
    report_data = []
    
    for agg_file in sorted(agg_files):
        filepath = os.path.join(aggregated_dir, agg_file)
        
        try:
            df = pd.read_csv(filepath)
            
            if df.empty:
                print(f"--- 文件: {agg_file} ---")
                print("  文件为空，跳过验证。\n")
                continue

            # 计算 estimateW 的总和
            sum_of_estimates = df['estimateW'].sum()
            
            # 获取 global_estimateW (取第一个非空值)
            global_estimate = df['global_estimateW'].dropna().iloc[0] if not df['global_estimateW'].dropna().empty else 0
            
            # 计算差异和比率
            difference = abs(sum_of_estimates - global_estimate)
            # 避免除以零的错误
            if global_estimate != 0:
                ratio = sum_of_estimates / global_estimate
            elif sum_of_estimates == 0:
                ratio = 1.0 # 如果两者都为0，则认为是一致的
            else:
                ratio = np.inf # 如果全局为0但加总不为0，则比率为无穷大

            # 准备报告内容
            result_entry = {
                "filename": agg_file,
                "sum_of_estimates": sum_of_estimates,
                "global_estimate": global_estimate,
                "difference": difference,
                "ratio": ratio
            }
            report_data.append(result_entry)

            # 打印单文件结果
            print(f"--- 文件: {agg_file} ---")
            print(f"  Sum(estimateW)      : {sum_of_estimates:,.4f}")
            print(f"  Global Estimate     : {global_estimate:,.4f}")
            print(f"  Absolute Difference : {difference:,.4f}")
            print(f"  Ratio (Sum / Global): {ratio:.6f}")
            if abs(ratio - 1.0) < 1e-4: # 设置一个很小的容忍度
                print("  状态: ✅ 一致")
            else:
                print("  状态: ❌ 不一致")
            print("-" * (len(agg_file) + 6))

        except Exception as e:
            print(f"--- 文件: {agg_file} ---")
            print(f"  处理时发生错误: {e}\n")

    # --- 3. 生成并保存总结报告 ---
    if not report_data:
        print("没有可用于生成报告的数据。")
        return

    report_df = pd.DataFrame(report_data)
    
    # 将报告保存为文本文件，格式化以便阅读
    with open(report_path, 'w') as f:
        f.write(f"一致性验证报告 for dataset: {dataset_name}\n")
        f.write("=" * 50 + "\n\n")
        # 使用 to_string() 方法可以很好地格式化输出
        f.write(report_df.to_string(index=False))
        
    print(f"\n[完成] 验证报告已保存到: {report_path}")

if __name__ == '__main__':
    # --- 配置区 ---
    dataset_to_process = 'dataset_test'
    verify_estimates_consistency(dataset_to_process)

多谓词执行pipeline - 运行分层采样方法

In [21]:
from pythonProject.src.Structure_first.proxy_sample import multi_predicate_evaluation
from pythonProject.src.Structure_first.proxy_sample import evaluate_graph_only_baseline
# ===========================
dataset_to_process = 'dataset_test'
multi_predicate_evaluation(dataset_to_process)


========== 开始对数据集 'dataset_test' 进行多谓词采样评估 ==========

>>> 步骤 1: 获取所有查询的 T_true...
✅ 找到 T_true 缓存文件: /home/wangshuo/resource/datasets/parler_data/dataset_test/results/T_true.json
已从缓存加载 T_true 数据。

>>> 步骤 2: 运行采样评估...


==================== 评估查询: query_cycle_4_0 ====================
  (使用 T_true = 12919.0)
[Check_total_budget_frac] 0.05
[警告] 运行次数 (runs=1) 小于等于2，无法移除最大和最小值。将按常规方式计算统计。
将为 7 种方法，每种运行 1 次...

--- 估算结果总结 ---
Method               | T_hat (mean ± std)        | Qerror (mean ± std)       | Samples(Post/Cmt) 
----------------------------------------------------------------------------------------------------
proxy_importance     |  14213.227 ± 0.000      |     0.1002 ± 0.0000     |    146 / 259   
proxy_uniform        |  12754.230 ± 0.000      |     0.0128 ± 0.0000     |    169 / 259   
proxyE_importance    |  11880.683 ± 0.000      |     0.0804 ± 0.0000     |    150 / 259   
proxyE_uniform       |  15282.096 ± 0.000      |     0.1829 ± 0.0000     |    163 / 259   
baseline_uni

In [22]:
evaluate_graph_only_baseline(dataset_to_process)


========== 开始对数据集 'dataset_test' 运行 [Graph Only] 基线评估 ==========
>>> 步骤 1: 获取 T_true...
✅ 找到 T_true 缓存文件: /home/wangshuo/resource/datasets/parler_data/dataset_test/results/T_true.json
已从缓存加载 T_true 数据。
>>> 步骤 2: 读取现有报告 /home/wangshuo/resource/datasets/parler_data/dataset_test/results/results_summary2.csv ...
>>> 步骤 3: 计算 Graph Only 基线...


Evaluating:   0%|                                                                            | 0/153 [00:00<?, ?query/s]

[Check_total_budget_frac] 0.05


Evaluating:   1%|▍                                                                   | 1/153 [00:01<04:07,  1.63s/query]

Query: query_cycle_4_0.graph          | T_hat:     13810 | Qerr: 0.0690 | Nodes: P=1731, C=5123
[Check_total_budget_frac] 0.05


...

Query: query_cycle_8_5.graph          | T_hat:    519858 | Qerr: 0.0843 | Nodes: P=1072, C=1613
[Check_total_budget_frac] 0.05


Evaluating:  96%|███████████████████████████████████████████████████████████████▍  | 147/153 [03:05<00:10,  1.82s/query]

Query: query_cycle_8_6.graph          | T_hat:      5614 | Qerr: 0.1608 | Nodes: P=717, C=3637
[Check_total_budget_frac] 0.05


Evaluating: 100%|██████████████████████████████████████████████████████████████████| 153/153 [03:06<00:00,  1.22s/query]

Query: query_cycle_8_8.graph          | T_hat:   1348719 | Qerr: 0.1185 | Nodes: P=1050, C=1187

✅ 已将 [baseline_graph_only] 结果追加到: /home/wangshuo/resource/datasets/parler_data/dataset_test/results/results_summary2.csv
    新增记录数: 113
    总记录数: 904
✅ 节点采样统计已追加到: /home/wangshuo/resource/datasets/parler_data/dataset_test/results/efficiency/sampled_node_count.csv
